In [5]:
PAK_CITIES = {
    "Karachi":  (24.8607, 67.0011),
    "Lahore":   (31.5204, 74.3587),
    "Islamabad":(33.6844, 73.0479),
    "Rawalpindi":(33.5651, 73.0169),
    "Faisalabad":(31.4180, 73.0790),
    "Multan":   (30.1575, 71.5249),
    "Peshawar": (34.0151, 71.5249),
    "Quetta":   (30.1798, 66.9750)
}


In [6]:
!pip install meteostat


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [7]:
from meteostat import Daily, Point
from datetime import datetime
import pandas as pd

cities = {
    "Karachi":  (24.8607, 67.0011),
    "Lahore":   (31.5204, 74.3587),
    "Islamabad":(33.6844, 73.0479),
    "Rawalpindi":(33.5651, 73.0169),
    "Faisalabad":(31.4180, 73.0790),
    "Multan":   (30.1575, 71.5249),
    "Peshawar": (34.0151, 71.5249),
    "Quetta":   (30.1798, 66.9750)
}

start = datetime(1960,1,1)
end   = datetime(2025,12,31)

all_data = []

for city, (lat, lon) in cities.items():
    print(f"Fetching {city}...")
    loc = Point(lat, lon)
    df = Daily(loc, start, end).fetch()

    df = df.reset_index()
    df["city"] = city

    all_data.append(df)

weather = pd.concat(all_data, ignore_index=True)
weather.to_csv("pakistan_city_weather_daily.csv", index=False)

print("Saved pakistan_city_weather_daily.csv")


Fetching Karachi...
Fetching Lahore...
Fetching Islamabad...
Fetching Rawalpindi...
Fetching Faisalabad...


Fetching Multan...


Fetching Peshawar...


Fetching Quetta...


Saved pakistan_city_weather_daily.csv


In [5]:
df = pd.read_csv("pakistan_city_weather_daily.csv")
df["time"] = pd.to_datetime(df["time"])

df["year"] = df["time"].dt.year
df["month"] = df["time"].dt.month

monthly = (
    df.groupby(["city","year","month"], as_index=False)
      .agg({
          "tavg":"mean",
          "tmin":"mean",
          "tmax":"mean",
          "wspd":"mean"
      })
)

monthly["heat_index_proxy"] = (
    0.5*monthly["tavg"] +
    0.3*monthly["tmax"] +
    0.2*monthly["tmin"]
)


In [ ]:
humidity = pd.read_csv("pakistan_humidity_daily.csv")
humidity["time"] = pd.to_datetime(humidity["time"] )

humidity["year"] = humidity["time"].dt.year
humidity["month"] = humidity["time"].dt.month

humidity_monthly = (
    humidity.groupby(["city", "year", "month"], as_index=False)
    .agg({
        "rh_max": "mean",
        "rh_min": "mean",
        "prcp": "mean",
        "et0": "mean"
    })
)

humidity_monthly["rh_avg"] = (
    0.5 * humidity_monthly["rh_max"] +
    0.5 * humidity_monthly["rh_min"]
)

humidity_monthly.head()